In [ ]:
!pip install -U -q "transformers>=4.52" accelerate torch torchaudio torchvision jiwer pandas tqdm torchcodec


# Granite Speech — Single Audio Playground

## Setup

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

MODEL_ID = "ibm-granite/granite-speech-5.0-2b"
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID, device_map=device,
    dtype=torch.bfloat16 if device == "cuda" else torch.float32,
).eval()

## Kaldi Test Data

In [3]:
import torchaudio
from IPython.display import Audio, display


dataset = []

dataset_path = "/training-1/voxpopuli/dataset_with_spaces_voxpopuli_en/test" #other languages *_de *_fr *_es
# other datasets
# Mozilla Commonvoice: /training-1/asr/kaldi_data/cv19/eb/test *_de *_fr *_es
# Mozilla Commonvoice Single Words: /training-1/asr/kaldi_data/singleword/de/test *_de *_fr *_es 
# Multilingual Librispeech: /training-1/asr/kaldi_data/mls/en/test *_de *_fr *_es

wav_scp_dict = {}
with open(f"{dataset_path}/wav.scp", encoding="UTF-8") as rf:
    for line in rf:
        id, wav_path = line.strip().split("\t")
        wav_scp_dict[id] = wav_path

text_dict = {}
with open(f"{dataset_path}/text", encoding="UTF-8") as rf:
    for line in rf:
        id, text = line.strip().split("\t")
        text_dict[id] = text

for id, wav_path in wav_scp_dict.items():
    text = text_dict[id]
    dataset.append({"audio" : wav_path, "text" : text})

first_entry = dataset[0]
print(first_entry)
print("-"*10)
print(first_entry["text"])
audio = torchaudio.load(first_entry["audio"])[0]
display(Audio(audio, rate=16000))



{'audio': '/training-1/asr/voxpopuli/out/transcribed_data/en/2013/20131022-0900-PLENARY-3-en_20131022-08:32:08_10.wav', 'text': 'then we as a parliament could take our responsibility and quickly vote through these measures because they will need to take effect on one january'}
----------
then we as a parliament could take our responsibility and quickly vote through these measures because they will need to take effect on one january


## Transcription Function (Prompt Changeable)

In [4]:
def transcribe(wav, prompt="can you transcribe the speech into a written format?", hotwords=None):
    if hotwords:
        prompt = f"{prompt}\nhotwords: {hotwords}"
    chat = [
        {"role": "system", "content": "You are Granite, developed by IBM. You are a helpful AI assistant."},
        {"role": "user", "content": f"<|audio|>{prompt}"},
    ]
    text = processor.tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = processor(text, wav, device=device, return_tensors="pt").to(device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=400, do_sample=False, num_beams=1)
    return processor.tokenizer.decode(out[0, inputs["input_ids"].shape[-1]:],
                                      skip_special_tokens=True).strip()

## Test on a Single Audio from the Test Dataset

In [7]:
import torchaudio

sample = dataset[0]

wav, sr = torchaudio.load(sample["audio"], normalize=True)
model_output = transcribe(wav, prompt="can you transcribe the speech into a written format?")

print("SAMPLE :", sample)
print("LABEL :", sample["text"])
print("MODEL_OUTPUT :", model_output)

SAMPLE : {'audio': '/training-1/asr/voxpopuli/out/transcribed_data/en/2013/20131022-0900-PLENARY-3-en_20131022-08:32:08_10.wav', 'text': 'then we as a parliament could take our responsibility and quickly vote through these measures because they will need to take effect on one january'}
LABEL : then we as a parliament could take our responsibility and quickly vote through these measures because they will need to take effect on one january
MODEL_OUTPUT : we as a parliament could take our responsibility and quickly vote through these measures because they need to take effect actually only one january


## Adding Hotwords

In [19]:

sample_path = "/training-1/asr/voxpopuli/out/transcribed_data/en/2018/20180704-0900-PLENARY-en_20180704-17:51:01_12.wav"
sample_text = "without these mistakes that you made we would not be debating this issue here today so what im asking mr kurz to do is to upscale this strategy to an eur one hundred and twenty million anti poverty strategy for all european citizens"

wav, sr = torchaudio.load(sample_path, normalize=True)
display(Audio(wav, rate=sr))

prompt = "can you transcribe the speech into a written format?"   # <-- change the prompt
hotwords = "mr kurz"   # e.g., "Siemens, Erlangen, Dr. Weber"

print(transcribe(wav, prompt=prompt, hotwords=None))
print("-"*10)
print(transcribe(wav, prompt=prompt, hotwords=hotwords))

with these mistakes that you guys made we would not be debating this issue here today so what i'm asking mr kool to do is to upscale this strategy to one hundred and twenty million anti poverty strategy for all european citizens
----------
with these mistakes that you guys made we would not be debating this issue here today so what i'm asking mr kurz to do is to upscale this strategy to one hundred and twenty million anti poverty strategy for all european citizens


## Compare Prompt Variants

In [20]:
prompts = [
    "can you transcribe the speech into a written format?",
    "Transcribe verbatim, including hesitations and filler words.",
    "Transcribe with proper punctuation and capitalization.",
]

for p in prompts:
    print("=" * 70)
    print("PROMPT:", p)
    print(transcribe(wav, prompt=p, hotwords=hotwords))

PROMPT: can you transcribe the speech into a written format?
with these mistakes that you guys made we would not be debating this issue here today so what i'm asking mr kool to do is to upscale this strategy to one hundred and twenty million anti poverty strategy for all european citizens
PROMPT: Transcribe verbatim, including hesitations and filler words.
with these mistakes that you guys made we would not be debating this issue here today so what i'm asking mr kool to do is to upscale this strategy to 120 million anti poverty strategy for all european citizens
PROMPT: Transcribe with proper punctuation and capitalization.
With these mistakes that you guys made, we would not be debating this issue here today. So what I'm asking Mr Kulcs to do is to upscale this strategy to 120 million anti poverty strategy for all European citizens.
